In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Demographics"
)

xpt_file = DATA_DIR / "DEMO_J.xpt"
csv_file = DATA_DIR / "DEMO_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2017-2018 DEMO_J XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SDDSRVYR",
    "RIDSTATR",
    "RIAGENDR",
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDRETH1",
    "RIDRETH3",
    "RIDEXMON",
    "RIDEXAGM",
    "DMQMILIZ",
    "DMQADFC",
    "DMDBORN4",
    "DMDCITZN",
    "DMDYRSUS",
    "DMDEDUC3",
    "DMDEDUC2",
    "DMDMARTL",
    "RIDEXPRG",
    "SIALANG",
    "SIAPROXY",
    "SIAINTRP",
    "FIALANG",
    "FIAPROXY",
    "FIAINTRP",
    "MIALANG",
    "MIAPROXY",
    "MIAINTRP",
    "AIALANGA",
    "DMDHHSIZ",
    "DMDFMSIZ",
    "DMDHHSZA",
    "DMDHHSZB",
    "DMDHHSZE",
    "DMDHRGND",
    "DMDHRAGZ",
    "DMDHREDZ",
    "DMDHRMAZ",
    "DMDHSEDZ",
    "WTINT2YR",
    "WTMEC2YR",
    "SDMVPSU",
    "SDMVSTRA",
    "INDHHIN2",
    "INDFMIN2",
    "INDFMPIR"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DEMO_J columns/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9254:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 46:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,254 rows x 46 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# TRUE DECIMAL VARIABLES
# ------------------------------------------------------------
#
# WTINT2YR
#     Full sample 2-year interview survey weight
#
# WTMEC2YR
#     Full sample 2-year MEC exam survey weight
#
# INDFMPIR
#     Family income-to-poverty ratio
#
#
# These must remain decimal.
#
# DO NOT:
#
#     round them
#     convert them to Int64
#     force decimal formatting
#
#
# All remaining DEMO variables are identifiers,
# categorical codes, ages/counts, or design variables
# and are numerically integral in this release.
#

DECIMAL_COLUMNS = [
    "WTINT2YR",
    "WTMEC2YR",
    "INDFMPIR"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 genuine decimal variables."
    )


if len(INTEGER_COLUMNS) != 43:

    raise ValueError(
        "Expected exactly 43 integer/code/count variables."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Integer/code/count fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. VERIFY ALL SOURCE VARIABLES ARE NUMERIC
# ============================================================

numeric_columns = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_columns != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 46 DEMO_J fields "
        "to load numerically."
    )


print(
    "PASS: All 46 variables loaded numerically."
)


# ============================================================
# 6. KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known XPORT decoded-zero artifact:
#
#     5.397605346934028e-79
#
#
# NEVER globally replace this without first validating
# where it occurs.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


ordinary_zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before repair:",
    f"{ordinary_zero_total:,}"
)

print(
    "Tiny XPORT values:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATED DEMO_J TINY-ZERO PATTERN
# ============================================================

#
# Exact attached-file pattern:
#
# RIDAGEYR     357
# RIDAGEMN      36
# RIDEXAGM      10
# DMDEDUC3     184
# DMDHHSZA    6183
# DMDHHSZB    4772
# DMDHHSZE    6125
# WTMEC2YR     550
# INDFMPIR      91
#
# TOTAL      18,308
#
#
# CDC independently confirms zero as valid:
#
# RIDAGEYR      range begins at 0
# RIDAGEMN      range begins at 0
# RIDEXAGM      range begins at 0
# DMDEDUC3      0 = kindergarten/never attended
# DMDHHSZA      explicit 0 count = 6183
# DMDHHSZB      explicit 0 count = 4772
# DMDHHSZE      explicit 0 count = 6125
# WTMEC2YR      0 = not MEC examined, count = 550
# INDFMPIR      published range begins at 0
#

EXPECTED_TINY_COUNTS = {
    "RIDAGEYR": 357,
    "RIDAGEMN": 36,
    "RIDEXAGM": 10,
    "DMDEDUC3": 184,
    "DMDHHSZA": 6183,
    "DMDHHSZB": 4772,
    "DMDHHSZE": 6125,
    "WTMEC2YR": 550,
    "INDFMPIR": 91
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: DEMO_J tiny-value pattern differs "
        "from the validated attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 18308:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before XPORT correction."
    )


print(
    "PASS: Exact 18,308-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 8. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Correct each validated variable independently.
#

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny count.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[col] = actual_count


total_corrections = sum(
    correction_counts.values()
)


print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


print(
    "Total restorations:",
    f"{total_corrections:,}"
)


if total_corrections != 18308:

    raise ValueError(
        "Total zero-restoration count incorrect."
    )


print(
    "PASS: 18,308 validated zeros restored."
)


# ============================================================
# 9. BUILD EXPECTED CORRECTED SOURCE
# ============================================================

#
# This dataframe represents exactly what is allowed
# to differ from the original source.
#
# ONLY:
#
#     validated tiny value -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 10. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny artifacts remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. FRACTIONAL VALUE DISCOVERY
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY as a comparison.
#
# It DOES NOT modify the dataframe.
#

print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_count = int(
            (
                ~np.isclose(
                    values,
                    np.round(values),
                    rtol=0,
                    atol=1e-12
                )
            )
            .sum()
        )


    fractional_counts[col] = (
        fractional_count
    )


for col, count in fractional_counts.items():

    if count > 0:

        print(
            f"{col}: {count:,}"
        )


EXPECTED_FRACTIONAL_COUNTS = {
    "WTINT2YR": 9254,
    "WTMEC2YR": 8704,
    "INDFMPIR": 6651
}


actual_fractional_columns = {
    col: count
    for col, count in fractional_counts.items()
    if count > 0
}


if (
    actual_fractional_columns
    !=
    EXPECTED_FRACTIONAL_COUNTS
):

    raise ValueError(
        "Unexpected fractional-value pattern.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{actual_fractional_columns}"
    )


print(
    "PASS: Only WTINT2YR, WTMEC2YR, and "
    "INDFMPIR contain genuine fractional values."
)


# ============================================================
# 12. PROVE INTEGER/CODE/COUNT FIELDS ARE INTEGRAL
# ============================================================

integer_fraction_failures = {}


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    failure_count = int(
        (~whole_mask)
        .sum()
    )


    if failure_count > 0:

        integer_fraction_failures[col] = (
            failure_count
        )


if integer_fraction_failures:

    raise ValueError(
        "STOP: Proposed integer/code fields "
        "contain fractional values:\n"
        + str(integer_fraction_failures)
    )


print(
    "PASS: All 43 proposed integer/code/count "
    "fields are integral."
)


# ============================================================
# 13. VALIDATE SEQN
# ============================================================

print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93703:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93703-102956 validated."
)


# ============================================================
# 14. BASIC CDC FREQUENCY CHECK HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        actual_missing
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 15. RELEASE CYCLE
# ============================================================

check_frequency(
    "SDDSRVYR",
    {
        10: 9254
    },
    0
)


# ============================================================
# 16. INTERVIEW / EXAMINATION STATUS
# ============================================================

check_frequency(
    "RIDSTATR",
    {
        1: 550,
        2: 8704
    },
    0
)


# ============================================================
# 17. GENDER
# ============================================================

check_frequency(
    "RIAGENDR",
    {
        1: 4557,
        2: 4697
    },
    0
)


# ============================================================
# 18. AGE IN YEARS
# ============================================================

#
# CDC:
#
#     0-79 = 8,827
#     80+  =   427
#
# In attached XPT:
#
#     age exactly 0 = 357
#
# represented by the tiny artifact before repair.
#

age_zero = int(
    df["RIDAGEYR"]
    .eq(0)
    .sum()
)


age_80 = int(
    df["RIDAGEYR"]
    .eq(80)
    .sum()
)


age_0_to_79 = int(
    df["RIDAGEYR"]
    .between(
        0,
        79,
        inclusive="both"
    )
    .sum()
)


print(
    "\n--- RIDAGEYR CHECK ---"
)

print(
    "Age 0:",
    age_zero
)

print(
    "Age 0-79:",
    age_0_to_79
)

print(
    "Age 80+:",
    age_80
)


if age_zero != 357:

    raise ValueError(
        "Unexpected RIDAGEYR zero count."
    )


if age_0_to_79 != 8827:

    raise ValueError(
        "RIDAGEYR 0-79 count differs from CDC."
    )


if age_80 != 427:

    raise ValueError(
        "RIDAGEYR age-80 count differs from CDC."
    )


print(
    "PASS: RIDAGEYR validated."
)


# ============================================================
# 19. AGE IN MONTHS AT SCREENING
# ============================================================

ridagemn_nonmissing = int(
    df["RIDAGEMN"]
    .notna()
    .sum()
)

ridagemn_missing = int(
    df["RIDAGEMN"]
    .isna()
    .sum()
)

ridagemn_zero = int(
    df["RIDAGEMN"]
    .eq(0)
    .sum()
)


print(
    "\n--- RIDAGEMN CHECK ---"
)

print(
    "Nonmissing:",
    ridagemn_nonmissing
)

print(
    "Missing:",
    ridagemn_missing
)

print(
    "Zero months:",
    ridagemn_zero
)

print(
    "Range:",
    df["RIDAGEMN"].min(),
    "to",
    df["RIDAGEMN"].max()
)


if ridagemn_nonmissing != 597:

    raise ValueError(
        "RIDAGEMN nonmissing count differs."
    )


if ridagemn_missing != 8657:

    raise ValueError(
        "RIDAGEMN missing count differs."
    )


if ridagemn_zero != 36:

    raise ValueError(
        "Unexpected RIDAGEMN zero count."
    )


if df["RIDAGEMN"].min() != 0:

    raise ValueError(
        "Unexpected RIDAGEMN minimum."
    )


if df["RIDAGEMN"].max() != 24:

    raise ValueError(
        "Unexpected RIDAGEMN maximum."
    )


print(
    "PASS: RIDAGEMN 0-24 months validated."
)


# ============================================================
# 20. AGE IN MONTHS AT EXAM
# ============================================================

ridexagm_nonmissing = int(
    df["RIDEXAGM"]
    .notna()
    .sum()
)

ridexagm_missing = int(
    df["RIDEXAGM"]
    .isna()
    .sum()
)

ridexagm_zero = int(
    df["RIDEXAGM"]
    .eq(0)
    .sum()
)


print(
    "\n--- RIDEXAGM CHECK ---"
)

print(
    "Nonmissing:",
    ridexagm_nonmissing
)

print(
    "Missing:",
    ridexagm_missing
)

print(
    "Zero months:",
    ridexagm_zero
)

print(
    "Range:",
    df["RIDEXAGM"].min(),
    "to",
    df["RIDEXAGM"].max()
)


if ridexagm_nonmissing != 3433:

    raise ValueError(
        "RIDEXAGM nonmissing count differs."
    )


if ridexagm_missing != 5821:

    raise ValueError(
        "RIDEXAGM missing count differs."
    )


if ridexagm_zero != 10:

    raise ValueError(
        "Unexpected RIDEXAGM zero count."
    )


if df["RIDEXAGM"].min() != 0:

    raise ValueError(
        "Unexpected RIDEXAGM minimum."
    )


if df["RIDEXAGM"].max() != 239:

    raise ValueError(
        "Unexpected RIDEXAGM maximum."
    )


print(
    "PASS: RIDEXAGM 0-239 months validated."
)


# ============================================================
# 21. CHILD/YOUTH EDUCATION ZERO CHECK
# ============================================================

#
# CDC explicitly:
#
#     DMDEDUC3 = 0
#     Never attended / kindergarten only
#     Count = 184
#

dmduc3_zero = int(
    df["DMDEDUC3"]
    .eq(0)
    .sum()
)


print(
    "\n--- DMDEDUC3 ZERO CHECK ---"
)

print(
    "Code 0:",
    dmduc3_zero
)


if dmduc3_zero != 184:

    raise ValueError(
        "DMDEDUC3 zero count differs from CDC."
    )


print(
    "PASS: DMDEDUC3 zero code validated."
)


# ============================================================
# 22. HOUSEHOLD COMPOSITION ZERO CHECKS
# ============================================================

#
# CDC explicitly reports:
#
# DMDHHSZA
#     0 young children = 6183
#
# DMDHHSZB
#     0 children age 6-17 = 4772
#
# DMDHHSZE
#     0 adults age 60+ = 6125
#

EXPECTED_HOUSEHOLD_ZERO_COUNTS = {
    "DMDHHSZA": 6183,
    "DMDHHSZB": 4772,
    "DMDHHSZE": 6125
}


EXPECTED_HOUSEHOLD_RANGES = {
    "DMDHHSZA": (0, 3),
    "DMDHHSZB": (0, 3),
    "DMDHHSZE": (0, 3)
}


print(
    "\n--- HOUSEHOLD COMPOSITION CHECK ---"
)


for col, expected_zero in (
    EXPECTED_HOUSEHOLD_ZERO_COUNTS.items()
):

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    expected_min, expected_max = (
        EXPECTED_HOUSEHOLD_RANGES[col]
    )


    print(
        f"{col}: "
        f"zero={zero_count:,}, "
        f"range={df[col].min()}-{df[col].max()}"
    )


    if zero_count != expected_zero:

        raise ValueError(
            f"{col} zero count differs."
        )


    if df[col].min() != expected_min:

        raise ValueError(
            f"{col} minimum differs."
        )


    if df[col].max() != expected_max:

        raise ValueError(
            f"{col} maximum differs."
        )


print(
    "PASS: Household composition zeros validated."
)


# ============================================================
# 23. SAMPLE WEIGHT CHECKS
# ============================================================

#
# Do NOT convert these to Int64.
#

print(
    "\n--- SAMPLE WEIGHT CHECK ---"
)


# ------------------------------------------------------------
# WTINT2YR
# ------------------------------------------------------------

wtint_nonmissing = int(
    df["WTINT2YR"]
    .notna()
    .sum()
)

wtint_missing = int(
    df["WTINT2YR"]
    .isna()
    .sum()
)

wtint_zero = int(
    df["WTINT2YR"]
    .eq(0)
    .sum()
)

wtint_fractional = (
    fractional_counts["WTINT2YR"]
)


wtint_min = float(
    df["WTINT2YR"]
    .min()
)

wtint_max = float(
    df["WTINT2YR"]
    .max()
)


print(
    "WTINT2YR:"
)

print(
    "  nonmissing:",
    wtint_nonmissing
)

print(
    "  missing:",
    wtint_missing
)

print(
    "  zeros:",
    wtint_zero
)

print(
    "  fractional observations:",
    wtint_fractional
)

print(
    "  range:",
    repr(wtint_min),
    "to",
    repr(wtint_max)
)


if wtint_nonmissing != 9254:

    raise ValueError(
        "WTINT2YR nonmissing count differs."
    )


if wtint_missing != 0:

    raise ValueError(
        "WTINT2YR contains missing values."
    )


if wtint_zero != 0:

    raise ValueError(
        "WTINT2YR unexpectedly contains zero."
    )


if wtint_fractional != 9254:

    raise ValueError(
        "Unexpected WTINT2YR decimal pattern."
    )


#
# CDC HTML displays rounded endpoints:
#
#     2571.0687123 - 433085.00526
#
# Allow small tolerance because the XPT retains more
# floating-point digits than the displayed HTML table.
#

if not np.isclose(
    wtint_min,
    2571.0687123,
    rtol=0,
    atol=1e-6
):

    raise ValueError(
        "WTINT2YR minimum differs from CDC."
    )


if not np.isclose(
    wtint_max,
    433085.00526,
    rtol=0,
    atol=1e-5
):

    raise ValueError(
        "WTINT2YR maximum differs from CDC."
    )


# ------------------------------------------------------------
# WTMEC2YR
# ------------------------------------------------------------

wtmec_nonmissing = int(
    df["WTMEC2YR"]
    .notna()
    .sum()
)

wtmec_missing = int(
    df["WTMEC2YR"]
    .isna()
    .sum()
)

wtmec_zero = int(
    df["WTMEC2YR"]
    .eq(0)
    .sum()
)

wtmec_positive_count = int(
    df["WTMEC2YR"]
    .gt(0)
    .sum()
)

wtmec_fractional = (
    fractional_counts["WTMEC2YR"]
)


wtmec_positive = (
    df.loc[
        df["WTMEC2YR"] > 0,
        "WTMEC2YR"
    ]
)


wtmec_min_positive = float(
    wtmec_positive.min()
)

wtmec_max = float(
    wtmec_positive.max()
)


print(
    "\nWTMEC2YR:"
)

print(
    "  positive:",
    wtmec_positive_count
)

print(
    "  zero:",
    wtmec_zero
)

print(
    "  missing:",
    wtmec_missing
)

print(
    "  fractional observations:",
    wtmec_fractional
)

print(
    "  positive range:",
    repr(wtmec_min_positive),
    "to",
    repr(wtmec_max)
)


if wtmec_nonmissing != 9254:

    raise ValueError(
        "WTMEC2YR nonmissing count differs."
    )


if wtmec_missing != 0:

    raise ValueError(
        "WTMEC2YR contains missing values."
    )


if wtmec_positive_count != 8704:

    raise ValueError(
        "WTMEC2YR positive-weight count differs."
    )


if wtmec_zero != 550:

    raise ValueError(
        "WTMEC2YR zero count differs from CDC."
    )


if wtmec_fractional != 8704:

    raise ValueError(
        "Unexpected WTMEC2YR fractional pattern."
    )


if not np.isclose(
    wtmec_min_positive,
    2566.1838545,
    rtol=0,
    atol=1e-6
):

    raise ValueError(
        "WTMEC2YR positive minimum differs."
    )


if not np.isclose(
    wtmec_max,
    419762.83649,
    rtol=0,
    atol=1e-5
):

    raise ValueError(
        "WTMEC2YR maximum differs."
    )


print(
    "PASS: Survey-weight distributions validated."
)


# ============================================================
# 24. WTMEC2YR ROUTING CHECK
# ============================================================

#
# CDC:
#
# RIDSTATR = 1
#     interviewed only = 550
#
# WTMEC2YR = 0
#     not MEC examined = 550
#
# Attachment shows exact row-level alignment.
#

not_examined = (
    df["RIDSTATR"]
    .eq(1)
    .to_numpy()
)


zero_mec_weight = (
    df["WTMEC2YR"]
    .eq(0)
    .to_numpy()
)


wtmec_route_mismatches = int(
    np.sum(
        not_examined
        !=
        zero_mec_weight
    )
)


print(
    "\n--- WTMEC2YR / RIDSTATR ROUTING ---"
)

print(
    "RIDSTATR=1:",
    int(
        not_examined.sum()
    )
)

print(
    "WTMEC2YR=0:",
    int(
        zero_mec_weight.sum()
    )
)

print(
    "Row-level mismatches:",
    wtmec_route_mismatches
)


if wtmec_route_mismatches != 0:

    raise ValueError(
        "WTMEC2YR zero values do not align "
        "with non-examined participants."
    )


print(
    "PASS: WTMEC2YR zero routing matches exactly."
)


# ============================================================
# 25. RIDEXMON / EXAMINATION STATUS CHECK
# ============================================================

#
# RIDEXMON is missing for exactly 550 people who
# were interviewed but never examined.
#

ridexmon_missing = (
    df["RIDEXMON"]
    .isna()
    .to_numpy()
)


ridexmon_route_mismatch = int(
    np.sum(
        ridexmon_missing
        !=
        not_examined
    )
)


print(
    "\n--- RIDEXMON ROUTING CHECK ---"
)

print(
    "RIDEXMON missing:",
    int(
        ridexmon_missing.sum()
    )
)

print(
    "Routing mismatches:",
    ridexmon_route_mismatch
)


if ridexmon_route_mismatch != 0:

    raise ValueError(
        "RIDEXMON missing positions do not "
        "match interview-only participants."
    )


print(
    "PASS: RIDEXMON exam routing validated."
)


# ============================================================
# 26. FAMILY INCOME-TO-POVERTY RATIO
# ============================================================

#
# INDFMPIR is a genuine decimal variable.
#
# CDC:
#
#     0 - 4.98    = 6815
#     >= 5 coded 5 = 1208
#     missing      = 1231
#
# Total nonmissing = 8023
#

pir_nonmissing = int(
    df["INDFMPIR"]
    .notna()
    .sum()
)

pir_missing = int(
    df["INDFMPIR"]
    .isna()
    .sum()
)

pir_zero = int(
    df["INDFMPIR"]
    .eq(0)
    .sum()
)

pir_five = int(
    df["INDFMPIR"]
    .eq(5)
    .sum()
)

pir_under_five = int(
    df["INDFMPIR"]
    .between(
        0,
        4.98,
        inclusive="both"
    )
    .sum()
)


print(
    "\n--- INDFMPIR CHECK ---"
)

print(
    "Nonmissing:",
    pir_nonmissing
)

print(
    "Missing:",
    pir_missing
)

print(
    "Zero:",
    pir_zero
)

print(
    "0-4.98:",
    pir_under_five
)

print(
    "Exactly 5:",
    pir_five
)

print(
    "Fractional observations:",
    fractional_counts["INDFMPIR"]
)

print(
    "Range:",
    df["INDFMPIR"].min(),
    "to",
    df["INDFMPIR"].max()
)


if pir_nonmissing != 8023:

    raise ValueError(
        "INDFMPIR nonmissing count differs."
    )


if pir_missing != 1231:

    raise ValueError(
        "INDFMPIR missing count differs."
    )


if pir_zero != 91:

    raise ValueError(
        "Unexpected INDFMPIR zero count."
    )


if pir_under_five != 6815:

    raise ValueError(
        "INDFMPIR 0-4.98 count differs from CDC."
    )


if pir_five != 1208:

    raise ValueError(
        "INDFMPIR value-5 count differs from CDC."
    )


if df["INDFMPIR"].min() != 0:

    raise ValueError(
        "INDFMPIR minimum differs."
    )


if df["INDFMPIR"].max() != 5:

    raise ValueError(
        "INDFMPIR maximum differs."
    )


if fractional_counts["INDFMPIR"] != 6651:

    raise ValueError(
        "Unexpected INDFMPIR fractional pattern."
    )


print(
    "PASS: INDFMPIR distribution validated."
)


# ============================================================
# 27. PROTECT ALL THREE DECIMAL VARIABLES
# ============================================================

print(
    "\n--- DECIMAL VARIABLE PROTECTION ---"
)


decimal_pre_conversion_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_pre_conversion_failures.append(
            col
        )


if decimal_pre_conversion_failures:

    raise ValueError(
        "Decimal variables changed unexpectedly:\n"
        + str(decimal_pre_conversion_failures)
    )


print(
    "PASS: WTINT2YR, WTMEC2YR, and INDFMPIR "
    "remain numerically unchanged apart from "
    "validated zero restorations."
)


# ============================================================
# 28. FINAL ZERO PATTERN BEFORE TYPE CONVERSION
# ============================================================

#
# Original XPT had zero ordinary numeric zeros.
#
# Therefore every final zero must correspond to one
# of the validated tiny-value restorations.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if count > 0:

        final_zero_counts[col] = count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if (
    final_zero_counts
    !=
    EXPECTED_TINY_COUNTS
):

    raise ValueError(
        "Final zero pattern differs from "
        "validated DEMO_J pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern matches exactly."
)


# ============================================================
# 29. PRE-CONVERSION COMPLETE FIDELITY
# ============================================================

pre_conversion_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_conversion_failures.append(
            col
        )


if pre_conversion_failures:

    raise ValueError(
        "Unexpected numeric changes before "
        "type conversion:\n"
        + str(pre_conversion_failures)
    )


print(
    "PASS: Corrected dataframe exactly matches "
    "expected corrected source."
)


# ============================================================
# 30. CONVERT ONLY PROVEN INTEGER/CODE/COUNT FIELDS
# ============================================================

#
# Do NOT touch:
#
#     WTINT2YR
#     WTMEC2YR
#     INDFMPIR
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} cannot safely become Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 43 ID/code/count fields converted "
    "to nullable Int64."
)

print(
    "PASS: 3 genuine decimal fields retained untouched."
)


# ============================================================
# 31. DECIMAL FIELD FIDELITY AFTER Int64 CONVERSION
# ============================================================

decimal_after_conversion_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    actual_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        actual_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_after_conversion_failures.append(
            col
        )


if decimal_after_conversion_failures:

    raise ValueError(
        "Decimal variables changed after "
        "integer conversion:\n"
        + str(decimal_after_conversion_failures)
    )


print(
    "PASS: Decimal variables still exactly preserved."
)


# ============================================================
# 32. COMPLETE PRE-EXPORT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    actual_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        actual_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export numeric fidelity failures:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 46 DEMO_J variables match exact "
    "expected corrected values."
)


# ============================================================
# 33. MISSING-POSITION FIDELITY BEFORE EXPORT
# ============================================================

missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_position_errors += int(
        (
            source_missing
            !=
            current_missing
        )
        .sum()
    )


print(
    "\nMissing-position differences:",
    missing_position_errors
)


if missing_position_errors != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 34. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY POLICY
# ------------------------------------------------------------
#
# DO NOT:
#
#     df.round(...)
#
# DO NOT:
#
#     float_format=
#
# DO NOT:
#
#     convert survey weights to integer
#
# DO NOT:
#
#     round INDFMPIR
#
#
# Genuine decimal values are serialized naturally from
# their source floating-point values.
#
# This is the same policy being used to protect genuine
# Dietary, Laboratory, Examination, weight, and exposure
# variables in the full NHANES workflow.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 35. RE-READ USING ROUND-TRIP FLOAT PARSER
# ============================================================

#
# Exact floating-point fidelity MUST be checked with:
#
#     float_precision="round_trip"
#
# rather than reparsing raw strings with pd.to_numeric().
#

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != (
    9254,
    46
):

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 36. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "Corrected XPT -> CSV fidelity failures:\n"
        + str(roundtrip_failures)
    )


print(
    "PASS: All 46 variables round-trip exactly."
)


# ============================================================
# 37. EXACT DECIMAL VARIABLE ROUND-TRIP
# ============================================================

print(
    "\n--- DECIMAL VARIABLE EXACT FIDELITY ---"
)


decimal_csv_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    print(
        f"{col}:",
        "PASS" if same else "FAIL"
    )


    if not same:

        decimal_csv_failures.append(
            col
        )


if decimal_csv_failures:

    raise ValueError(
        "Decimal round-trip failures:\n"
        + str(decimal_csv_failures)
    )


print(
    "PASS: All three genuine decimal variables "
    "round-trip exactly."
)


# ============================================================
# 38. CSV MISSING-POSITION FIDELITY
# ============================================================

csv_missing_errors = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    csv_missing_errors += int(
        (
            expected_missing
            !=
            exported_missing
        )
        .sum()
    )


if csv_missing_errors != 0:

    raise ValueError(
        "CSV missing-value positions changed."
    )


print(
    "PASS: CSV missing positions preserved exactly."
)


# ============================================================
# 39. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Raw CSV text is used ONLY for formatting checks.
#
# It is NOT used for exact binary-float equality.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    9254,
    46
):

    raise ValueError(
        "Raw CSV dimensions changed."
    )


if (
    raw_csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "Raw CSV column order changed."
    )


# ============================================================
# 40. INTEGER/CODE FIELD ".0" CHECK
# ============================================================

#
# ONLY the 43 proven integer/code/count variables.
#
# Do NOT apply this to:
#
#     WTINT2YR
#     WTMEC2YR
#     INDFMPIR
#

print(
    "\n--- INTEGER/CODE/COUNT .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = (
        raw_csv_df[col]
    )


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' found in "
        "integer/code/count fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "integer/code/count fields."
)


# ============================================================
# 41. SEQN TEXT CHECK
# ============================================================

if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "SEQN still contains '.0'."
    )


print(
    "PASS: SEQN contains no '.0'."
)


# ============================================================
# 42. DECIMAL RAW-TEXT POLICY
# ============================================================

#
# Decimal variables are NOT checked for ".0".
#
# For example:
#
#     INDFMPIR = 5.0
#
# can legitimately serialize with ".0".
#
# The VARIABLE itself remains a decimal measure.
#

print(
    "\n--- DECIMAL FIELD RAW-TEXT CHECK ---"
)


for col in DECIMAL_COLUMNS:

    populated = int(
        raw_csv_df[col]
        .ne("")
        .sum()
    )


    dot_zero = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    print(
        f"{col}: "
        f"populated={populated:,}, "
        f"tokens ending .0={dot_zero:,}"
    )


print(
    "PASS: Decimal fields retained without "
    "cosmetic integer conversion."
)


# ============================================================
# 43. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_total += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY-VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_total
)


if tiny_csv_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 44. FINAL ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if (
    csv_zero_counts
    !=
    EXPECTED_TINY_COUNTS
):

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final zero pattern matches all "
    "18,308 validated restorations."
)


# ============================================================
# 45. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_notation_errors = {}


for col in EXPECTED_COLUMNS:

    tokens = (
        raw_csv_df[col]
    )


    count = int(
        tokens
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


    if count > 0:

        scientific_notation_errors[col] = (
            count
        )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    scientific_notation_errors
)


if scientific_notation_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(scientific_notation_errors)
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 46. FINAL SEQN VALIDATION
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93703:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final SEQN contains missing values."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final SEQN contains duplicates."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final SEQN contains '.0'."
    )


print(
    "PASS: Final SEQN 93703-102956 valid "
    "with no '.0'."
)


# ============================================================
# 47. ROUND-TRIP MAXIMUM ABSOLUTE DIFFERENCE
# ============================================================

#
# Expected:
#
#     0.0
#
# for every variable.
#

print(
    "\n--- ROUND-TRIP MAXIMUM ABSOLUTE DIFFERENCES ---"
)


difference_failures = {}


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    valid_mask = (
        ~np.isnan(expected_values)
        &
        ~np.isnan(final_values)
    )


    if valid_mask.any():

        max_difference = float(
            np.max(
                np.abs(
                    expected_values[valid_mask]
                    -
                    final_values[valid_mask]
                )
            )
        )

    else:

        max_difference = 0.0


    print(
        f"{col}: {max_difference}"
    )


    if max_difference != 0:

        difference_failures[col] = (
            max_difference
        )


if difference_failures:

    raise ValueError(
        "Round-trip numeric differences detected:\n"
        + str(difference_failures)
    )


print(
    "PASS: Maximum numeric difference = 0 "
    "for all 46 variables."
)


# ============================================================
# 48. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DEMO_J CONVERSION COMPLETE"
)

print(
    "=" * 100
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Demographics"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)

print(
    "Validated zero restorations:",
    f"{total_corrections:,}"
)

print(
    "Integer/code/count fields converted to Int64:",
    len(INTEGER_COLUMNS)
)

print(
    "Genuine decimal fields preserved:",
    DECIMAL_COLUMNS
)

print(
    "Interview-only participants:",
    int(
        df["RIDSTATR"]
        .eq(1)
        .sum()
    )
)

print(
    "MEC-examined participants:",
    int(
        df["RIDSTATR"]
        .eq(2)
        .sum()
    )
)

print(
    "WTMEC2YR zero weights:",
    wtmec_zero
)

print(
    "INDFMPIR zero values:",
    pir_zero
)

print(
    "INDFMPIR nonmissing:",
    pir_nonmissing
)

print(
    "INDFMPIR range:",
    f"{df['INDFMPIR'].min()}-"
    f"{df['INDFMPIR'].max()}"
)

print(
    "Exact decimal round-trip failures:",
    0
)

print(
    "Exact all-column round-trip failures:",
    0
)

print(
    "Missing-position differences:",
    0
)

print(
    "Tiny values remaining:",
    0
)

print(
    "Unnecessary '.0' in integer/code fields:",
    0
)

print(
    "Scientific notation:",
    0
)

print(
    "=" * 100
)

NHANES 2017-2018 DEMO_J XPT -> CSV
Rows: 9,254
Columns: 46
PASS: Structure = 9,254 rows x 46 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['WTINT2YR', 'WTMEC2YR', 'INDFMPIR']
Integer/code/count fields: 43
PASS: All 46 variables loaded numerically.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before repair: 0
Tiny XPORT values: 18,308

Tiny values by variable:
DMDHHSZA    6183
DMDHHSZE    6125
DMDHHSZB    4772
WTMEC2YR     550
RIDAGEYR     357
DMDEDUC3     184
INDFMPIR      91
RIDAGEMN      36
RIDEXAGM      10
PASS: Exact 18,308-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
RIDAGEYR: 357
RIDAGEMN: 36
RIDEXAGM: 10
DMDEDUC3: 184
DMDHHSZA: 6,183
DMDHHSZB: 4,772
DMDHHSZE: 6,125
WTMEC2YR: 550
INDFMPIR: 91
Total restorations: 18,308
PASS: 18,308 validated zeros restored.

Tiny artifacts remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE DISCOVERY ---
WTINT2YR: 9,254
WTMEC2YR: 8,704
INDFMPIR: 6,651
PASS: Only WTINT2YR, WTME